# 31 DPO evaluation (CPU runtime)

Scores the DPO model on validation, dev and mini-dev, and compares it on validation with the SFT model it started from, by the same rule as the SFT models: DPO is kept only if its validation EX is not significantly below the better of the two (paired McNemar test) and its validation R-VES is higher. Dev and mini-dev are reported, never used to choose.

**Runtime > Change runtime type > CPU** (timing runs on the same machine type as all earlier timing).

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = False  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
import json, os
POOL, EXPORT, DPO_EXPORT = "pool_v1", "sft_v1", "dpo_v1"
R = f"{DRIVE_ROOT}/effsql/runs"
# The SFT model kept on validation by R-VES (25_sft_evaluate: method, then epoch 1 or 2): DPO starts from it.
CHOSEN = json.load(open(f"{R}/{EXPORT}_epoch_choice_rves/selection.json"))["winner"]
cm = json.load(open(f"{R}/{CHOSEN}/adapter_meta.json"))
INIT, TAG = cm.get("adapter_run", CHOSEN), cm.get("tag")  # TAG only for a tagged model
TAG_ARG = f"--tag={TAG}" if TAG else ""
METHOD = f"dpo-{cm['arm']}" + (f"-tag{TAG}" if TAG else "")
RUN = f"{DPO_EXPORT}_{cm['arm']}"
print("SFT model:", CHOSEN, "| adapter:", INIT, "| target:", TAG, "| DPO run:", RUN)
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
meta = json.load(open(f"{R}/{EXPORT}/export_meta.json"))
VAL_DBS = ",".join(meta["holdout_db_ids"])

### 1. Stage the validation databases and the dev databases

In [ ]:
!effsql stage --split train --db-ids {VAL_DBS}
!effsql stage --split dev

### 2. Score the DPO model (resumable)

In [ ]:
!python -m effsql.train eval-val --run {RUN}
!effsql evaluate --pred {R}/{RUN}/predictions_dev.jsonl --split dev --timing none --run {RUN}_dev --model {MODEL} --method {METHOD}
!effsql evaluate --pred {R}/{RUN}/predictions_minidev.jsonl --split minidev --timing quick --run {RUN}_minidev --model {MODEL} --method {METHOD}

### 3. DPO against its SFT starting point on validation

In [ ]:
!python -m effsql.train select --runs {CHOSEN},{RUN} --out {DPO_EXPORT}_choice

### 4. Validation pair metrics during training

In [ ]:
import pandas as pd
print(pd.read_json(f"{R}/{RUN}/val_log.jsonl", lines=True).to_string(index=False))

### 5. All results: validation, dev and mini-dev, each model against the SFT starting point

Main metric: **R-VES** (BIRD's reward per question with step counts in place of seconds: at least 2x faster than gold 112, 1 to 2x 100, up to 2x slower 87, 2 to 4x slower 71, slower 50, wrong 0; gold = 100), with a 95% bootstrap interval. **R-VES/EX**: the same over correct answers only, the efficiency apart from accuracy. **VES** (100 x square root of the speed-up, unbounded) is shown but a few extreme questions drive it. Then each model against the reference on the same questions: R-VES difference (Wilcoxon test), EX (McNemar test), and how often its query is cheaper or costlier by at least 10% when both are correct (sign test). Nothing is executed: the table reads the saved evaluations. Saved as `report.txt` and `report.json`.

In [ ]:
REPORT = [CHOSEN, "pool_v1_{split}_base", f"{EXPORT}_gold_e1", RUN]
LABELS = ",".join(["pool_v1_{split}_base=zero-shot"] + [f"{r}={r.replace(EXPORT + '_', '')}" for r in REPORT if "{split}" not in r])
REF, OUT = CHOSEN, f"{DPO_EXPORT}_report"
!python -m effsql.train report --runs "{','.join(REPORT)}" --ref {REF} --labels "{LABELS}" --out {OUT}

### Done: disconnect the runtime